# CON / INT·WIS 50:50

## tl;dr

추가보너스의 가중치는 INT50%·WIS50%. CON은모험시간. 기획조건부권고이며 앱·DB에는미적용.

## Context & Methods

### Key Assumptions

P0직업적성식유지,최종상한마법사35%/다른직업30%. 합성게임144경로. 주비교96경로와추가48경로를분리해평가. 시간저장효율은캘린더환산모델.

일반Python순차실행검증. nbformat/nbclient/ipykernel미설치로Jupyter커널미실행. 커널환경에서 `python -m jupyter nbconvert --execute --to notebook --inplace con-mental-review.ipynb`로재검증가능.


## Data

### 1. Formula parity


In [1]:
from pathlib import Path
import sys,json,pandas as pd,numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'tools/analysis/con_mental_review.py').exists())
sys.path.insert(0,str(root/'tools/analysis'))
from con_mental_review import OUT,effects,verify
a=verify(pd.read_csv(OUT/'matched.csv'),list(range(14,30)))
b=verify(pd.read_csv(OUT/'heldout.csv'),list(range(30,38)))
print('Kotlin/Python rows:',len(a)+len(b))
assert len(a)+len(b)==15360


Kotlin/Python rows: 15360


## Results

### 2. Level100 and cap difficulty


In [2]:
z=pd.concat([a,b]);v=z[z.level==100]
print(v.groupby('class')[['cap_h','raw_proc_pct','search_s','sale_bonus_pct']].mean().round(4).to_string())
assert (v[v['class']=='MAGE'].raw_proc_pct<35).all()
first=b[(b['class']=='MAGE') & np.isclose(b.raw_proc_pct,35,atol=1e-10,rtol=0)].groupby('seed_index')['level'].min()
assert len(first)==8 and first.min()==106 and first.max()==112
print('Additional Mage paths first cap:',first.to_dict())


          cap_h  raw_proc_pct  search_s  sale_bonus_pct
class                                                  
CLERIC   9.0219       27.8966    4.6644         19.9167
MAGE     9.0219       33.2469    4.6644          6.6500
PALADIN  9.0073       23.1480    4.6606         19.9500
RANGER   9.0213       27.8665    4.0022          6.6333
ROGUE    9.0674       23.0759    4.0039          6.2056
WARRIOR  9.9949       23.1480    4.6606          6.3833
Additional Mage paths first cap: {30: 108, 31: 106, 32: 110, 33: 112, 34: 108, 35: 108, 36: 112, 37: 108}


### 3. Balance guardrails and safety


In [3]:
s=json.loads((OUT/'FINAL_SUMMARY.json').read_text())
for cohort in ['matched','heldout']:
 r=s[cohort]
 assert r['max_growth_spread_pct']<6 and r['max_sale_spread_pct']<12
 print(cohort,r['max_growth_spread_pct'],r['max_sale_spread_pct'])
assert '3840 PASS' in s['boundary_checks']
assert s['safety']['app_sources_unchanged'] and not s['safety']['app_start']
print(s['safety'])


matched 5.55317549677572 10.893306106304923
heldout 5.889252307056858 10.657212766151613
{'network': 'OS denied', 'database_files': 'OS denied', 'device_registration': False, 'app_start': False, 'app_sources_unchanged': True}


## Takeaways

100레벨일반판정확률평균은마법사33.25%,성직자·레인저27.9%,물리직23.1%. 8시간완충은마법사/레인저가경쟁,긴간격/부분충전은전사,판매수입은성직자. 실측이용자성과나절대최적해를주장하지않는다.